# Fog-aware detection training pipeline

## 0. Install dependencies

In [5]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Config


In [6]:
import os
from pathlib import Path

CONFIG = {
    # raw data: dataset root containing train/ valid/ test/ subfolders
    "DATA_DIR": "G:\Projects\Low Visibility\low visibility Final.v2i.coco",                       
    "COCO_JSON_NAME": "_annotations.coco.json",    

    "SPLIT_DIRS": {"train": "train", "val": "valid", "test": "test"},

    # working dirs
    "YOLO_DATASET_DIR": "data/yolo_dataset",       
    "FOG_MODEL_PATH": "fog_model.pkl",

    # fog scoring (matches the earlier fog_classifier.py) ---
    "SCORE_RESOLUTION": (256, 256),
    "DARK_CHANNEL_PATCH": 15,

    # preprocessing ablation toggles
    "APPLY_DCP": True,
    "APPLY_EXPOSURE": True,

    # training image size (final resize; must match inference pipeline)
    "IMG_SIZE": 640,
    "RESIZE_INTERPOLATION": "INTER_AREA",          

    "SEED": 42,

    # YOLO training
    "YOLO_WEIGHTS": "yolov8n.pt",                  
    "EPOCHS": 100,                                   
    "BATCH": 16,


    # device
    "DEVICE": 0,  
}

Path(CONFIG["YOLO_DATASET_DIR"]).mkdir(parents=True, exist_ok=True)

## 2. Load COCO annotations for each split


In [7]:
from pycocotools.coco import COCO

coco_by_split = {}
image_dir_by_split = {}
for split, subdir in CONFIG["SPLIT_DIRS"].items():
    split_dir = Path(CONFIG["DATA_DIR"]) / subdir
    json_path = split_dir / CONFIG["COCO_JSON_NAME"]
    coco_by_split[split] = COCO(str(json_path))
    image_dir_by_split[split] = str(split_dir)
    print(f"{split}: {len(coco_by_split[split].getImgIds())} images")

cat_ids = sorted(coco_by_split["train"].getCatIds())
cat_id_to_name = {c["id"]: c["name"] for c in coco_by_split["train"].loadCats(cat_ids)}
cat_id_to_yolo_idx = {cid: i for i, cid in enumerate(cat_ids)}
yolo_idx_to_name = {i: cat_id_to_name[cid] for cid, i in cat_id_to_yolo_idx.items()}
print(f"classes: {list(cat_id_to_name.values())}")

# sanity check: make sure valid/test use the same category id -> name mapping as train
for split in ["val", "test"]:
    other_cats = {c["id"]: c["name"] for c in coco_by_split[split].loadCats(coco_by_split[split].getCatIds())}
    if other_cats != cat_id_to_name:
        print(f"[warn] category mapping in '{split}' differs from 'train': {other_cats} vs {cat_id_to_name}")

loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
train: 7695 images
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
val: 1847 images
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
test: 1380 images
classes: ['Bike-car-person', 'Bike', 'Car', 'Person']


## 3. Fog / no-fog feature extraction (unsupervised)


In [10]:
import cv2
import numpy as np
import pandas as pd
import joblib
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def estimate_fog_features(img_bgr, size=CONFIG["SCORE_RESOLUTION"], patch=CONFIG["DARK_CHANNEL_PATCH"]):
    img = cv2.resize(img_bgr, size, interpolation=cv2.INTER_AREA)
    min_channel = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    dark_channel = cv2.erode(min_channel, kernel)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return {
        "mean_dark_intensity": float(np.mean(dark_channel)),
        "laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        "mean_saturation": float(np.mean(hsv[:, :, 1])),
    }

rows = []
for split, coco in coco_by_split.items():
    image_dir = image_dir_by_split[split]
    for img_info in coco.loadImgs(coco.getImgIds()):
        path = str(Path(image_dir) / img_info["file_name"])
        img = cv2.imread(path)
        if img is None:
            print(f"[warn] could not read {path}")
            continue
        feats = estimate_fog_features(img)
        feats["split"] = split
        feats["image_id"] = img_info["id"]
        feats["path"] = path
        feats["file_name"] = img_info["file_name"]
        rows.append(feats)

fog_df = pd.DataFrame(rows)
print(f"Scored {len(fog_df)} images across all splits")

Scored 10922 images across all splits


In [11]:
feature_cols = ["mean_dark_intensity", "laplacian_var", "mean_saturation"]
scaler = StandardScaler()
X = scaler.fit_transform(fog_df[feature_cols])

km = KMeans(n_clusters=2, n_init=10, random_state=CONFIG["SEED"])
fog_df["cluster"] = km.fit_predict(X)

cluster_means = fog_df.groupby("cluster")["mean_dark_intensity"].mean()
fog_cluster = cluster_means.idxmax()  # higher dark-channel mean -> fog
fog_df["fog_label"] = (fog_df["cluster"] == fog_cluster).map({True: "fog", False: "clear"})

print(fog_df.groupby("split")["fog_label"].value_counts())
joblib.dump({"scaler": scaler, "kmeans": km, "fog_cluster": fog_cluster}, CONFIG["FOG_MODEL_PATH"])

split  fog_label
test   fog           948
       clear         432
train  fog          5443
       clear        2252
val    fog          1332
       clear         515
Name: count, dtype: int64


['fog_model.pkl']

## 4. Dark Channel Prior dehazing


In [12]:
def dark_channel(img, patch=15):
    min_channel = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    return cv2.erode(min_channel, kernel)

def estimate_atmospheric_light(img, dark, top_percent=0.001):
    h, w = dark.shape
    n_pixels = max(int(h * w * top_percent), 1)
    flat_dark = dark.reshape(-1)
    flat_img = img.reshape(-1, 3)
    idx = np.argpartition(flat_dark, -n_pixels)[-n_pixels:]
    # among the brightest dark-channel pixels, take the one with highest intensity in the original image
    brightest = idx[np.argmax(flat_img[idx].sum(axis=1))]
    return flat_img[brightest].astype(np.float64)

def guided_filter(guide, src, radius=40, eps=1e-3):
    guide = guide.astype(np.float64)
    src = src.astype(np.float64)
    mean_g = cv2.boxFilter(guide, cv2.CV_64F, (radius, radius))
    mean_s = cv2.boxFilter(src, cv2.CV_64F, (radius, radius))
    mean_gs = cv2.boxFilter(guide * src, cv2.CV_64F, (radius, radius))
    cov_gs = mean_gs - mean_g * mean_s
    mean_gg = cv2.boxFilter(guide * guide, cv2.CV_64F, (radius, radius))
    var_g = mean_gg - mean_g * mean_g
    a = cov_gs / (var_g + eps)
    b = mean_s - a * mean_g
    mean_a = cv2.boxFilter(a, cv2.CV_64F, (radius, radius))
    mean_b = cv2.boxFilter(b, cv2.CV_64F, (radius, radius))
    return mean_a * guide + mean_b

def estimate_transmission(img, A, patch=15, omega=0.95):
    normalized = img.astype(np.float64) / A
    t = 1 - omega * dark_channel(normalized, patch)
    return t

def dehaze(img_bgr, patch=15, omega=0.95, t0=0.1, guide_radius=40, guide_eps=1e-3):
    img = img_bgr.astype(np.float64)
    dark = dark_channel(img, patch)
    A = estimate_atmospheric_light(img, dark)
    t = estimate_transmission(img, A, patch, omega)
    gray_guide = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float64) / 255.0
    t_refined = guided_filter(gray_guide, t, radius=guide_radius, eps=guide_eps)
    t_refined = np.clip(t_refined, t0, 1.0)
    J = np.empty_like(img)
    for c in range(3):
        J[:, :, c] = (img[:, :, c] - A[c]) / t_refined + A[c]
    return np.clip(J, 0, 255).astype(np.uint8)

## 5. Exposure enhancement (scikit-image)


In [13]:
from skimage import exposure, color

def enhance_exposure(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB) / 255.0
    lab = color.rgb2lab(rgb)
    l_channel = lab[:, :, 0] / 100.0  # normalize L to [0, 1] for equalize_adapthist
    l_eq = exposure.equalize_adapthist(l_channel, clip_limit=0.01)
    lab[:, :, 0] = l_eq * 100.0
    rgb_eq = np.clip(color.lab2rgb(lab), 0, 1)
    return cv2.cvtColor((rgb_eq * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

## 6. Build the preprocessed dataset, per split


In [14]:
from tqdm import tqdm

interp = getattr(cv2, CONFIG["RESIZE_INTERPOLATION"])
yolo_root = Path(CONFIG["YOLO_DATASET_DIR"])

fog_label_by_key = {(r.split, r.image_id): r.fog_label for r in fog_df.itertuples()}
preprocessed_paths = {}  # (split, image_id) -> output path

for split, coco in coco_by_split.items():
    out_img_dir = yolo_root / "images" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    image_dir = image_dir_by_split[split]

    for img_info in tqdm(coco.loadImgs(coco.getImgIds()), desc=split):
        img_id = img_info["id"]
        src_path = str(Path(image_dir) / img_info["file_name"])
        img = cv2.imread(src_path)
        if img is None:
            continue

        if fog_label_by_key.get((split, img_id)) == "fog":
            if CONFIG["APPLY_DCP"]:
                img = dehaze(img)
            if CONFIG["APPLY_EXPOSURE"]:
                img = enhance_exposure(img)
        # clear images (or fog images with both toggles off) pass through untouched

        img = cv2.resize(img, (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), interpolation=interp)

        out_path = out_img_dir / img_info["file_name"]
        out_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), img)
        preprocessed_paths[(split, img_id)] = str(out_path)

train:   0%|          | 13/7695 [00:04<46:00,  2.78it/s]C:\Users\Ebrahim\AppData\Local\Temp\ipykernel_31792\1413642277.py:9: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 10 negative Z values that have been clipped to zero
  rgb_eq = np.clip(color.lab2rgb(lab), 0, 1)
train:   0%|          | 37/7695 [00:11<38:50,  3.29it/s]C:\Users\Ebrahim\AppData\Local\Temp\ipykernel_31792\1413642277.py:9: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 6 negative Z values that have been clipped to zero
  rgb_eq = np.clip(color.lab2rgb(lab), 0, 1)
train:   3%|▎         | 222/7695 [01:01<35:38,  3.49it/s]C:\Users\Ebrahim\AppData\Local\Temp\ipykernel_31792\1413642277.py:9: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 1 negative Z values that have been clipped to zero
  rgb_eq = np.clip(color.lab2rgb(lab), 0, 1)
train:   9%|▊         | 672/7695 [02:54<18:51,  6.21it/s]C:\Users\Ebrahim\AppData\Local\Temp\ipykerne

## 7. Convert COCO annotations to YOLO format, per split


In [15]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    return cx, cy, w / img_w, h / img_h

for split, coco in coco_by_split.items():
    out_label_dir = yolo_root / "labels" / split
    out_label_dir.mkdir(parents=True, exist_ok=True)

    for img_info in coco.loadImgs(coco.getImgIds()):
        img_id = img_info["id"]
        if (split, img_id) not in preprocessed_paths:
            continue
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        lines = []
        for ann in anns:
            cx, cy, w, h = coco_bbox_to_yolo(ann["bbox"], img_info["width"], img_info["height"])
            yolo_idx = cat_id_to_yolo_idx[ann["category_id"]]
            lines.append(f"{yolo_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        label_path = out_label_dir / (Path(img_info["file_name"]).stem + ".txt")
        label_path.write_text("\n".join(lines))

    print(f"{split}: wrote {len(list(out_label_dir.glob('*.txt')))} label files")

train: wrote 7695 label files
val: wrote 1847 label files
test: wrote 1380 label files


## 8. Write `data.yaml`

In [8]:
import yaml

yolo_root = Path(CONFIG["YOLO_DATASET_DIR"])
names = [yolo_idx_to_name[i] for i in range(len(yolo_idx_to_name))]
data_yaml = {
    "path": str(yolo_root.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: n for i, n in enumerate(names)},
}

data_yaml_path = yolo_root / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(data_yaml_path.read_text())

path: G:\Projects\Low Visibility\data\yolo_dataset
train: images/train
val: images/val
test: images/test
names:
  0: Bike-car-person
  1: Bike
  2: Car
  3: Person



## 10. Train YOLO (50 epochs for iteration runs — set `EPOCHS` to 100 for the final model)

In [6]:
from ultralytics import YOLO

model = YOLO(CONFIG["YOLO_WEIGHTS"])

train_kwargs = dict(
    data=str(data_yaml_path),
    epochs=CONFIG["EPOCHS"],
    imgsz=CONFIG["IMG_SIZE"],
    batch=CONFIG["BATCH"],
    seed=CONFIG["SEED"],
    device=CONFIG["DEVICE"],
    patience = 7,
    workers=2,
    plots = False,
    name=f"dcp{int(CONFIG['APPLY_DCP'])}_exp{int(CONFIG['APPLY_EXPOSURE'])}",
)


results = model.train(**train_kwargs)

New https://pypi.org/project/ultralytics/8.4.165 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

RuntimeError: DataLoader worker (pid(s) 26432) exited unexpectedly

In [3]:
from ultralytics import YOLO
model = YOLO("G:runs\detect\dcp1_exp1-9\weights\last.pt")

model.train(
    resume = True,
    workers = 2
)

New https://pypi.org/project/ultralytics/8.4.165 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000213361F3410>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [9]:
from ultralytics import YOLO
best = YOLO("runs/detect/dcp1_exp1-9/weights/best.pt")
test_metrics = best.val(data=str(data_yaml_path), split="val", device=CONFIG["DEVICE"], workers = 2)
print(test_metrics.box.map, test_metrics.box.map50)

Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 512.6194.2 MB/s, size: 101.1 KB)
val: Scanning G:\Projects\Low Visibility\data\yolo_dataset\labels\val.cache... 862 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 862/862  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.8it/s 19.6s0.2ss
                   all        862       2710      0.822      0.845       0.88      0.607
                  Bike        108        156      0.769      0.782      0.838      0.527
                   Car        636       1840      0.876      0.898      0.928      0.676
                Person        326        714      0.821      0.856      0.876      0.619
Speed: 1.6ms preprocess, 3.9ms inference, 0.0ms loss, 3.1ms postprocess per image
Results save

In [12]:
from ultralytics import YOLO
best = YOLO("runs/detect/dcp1_exp1-9/weights/best.pt")


# Evaluate on test set
metrics = best.val(
    data=str(data_yaml_path),
    split="test",
    device=CONFIG["DEVICE"]
)

# Display metrics
print("\n===== YOLO TEST RESULTS =====")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")
print(f"mAP@50        : {metrics.box.map50:.4f}")
print(f"mAP@50-95     : {metrics.box.map:.4f}")

Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 629.5306.5 MB/s, size: 129.8 KB)
val: Scanning G:\Projects\Low Visibility\data\yolo_dataset\labels\test.cache... 638 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 638/638  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 1.1it/s 37.6s0.2s21
                   all        638       2024      0.793      0.874      0.879      0.602
                  Bike         57         83      0.721      0.843      0.818      0.518
                   Car        453       1405      0.834      0.926      0.918      0.668
                Person        240        536      0.823      0.852        0.9      0.621
Speed: 1.7ms preprocess, 4.2ms inference, 0.0ms loss, 3.1ms postprocess per image
Results sa